<a href="https://colab.research.google.com/github/DivyaSwamy/transformers_tutorials/blob/main/FineTuning_ObjectDetction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### WIP - Following tutorial to learn about fine tuning a preexisting model.

https://huggingface.co/learn/cookbook/en/fine_tuning_detr_custom_dataset


#### Install Dependencies

In [1]:
!pip install -U -q datasets transformers[torch] timm wandb torchmetrics matplotlib albumentations


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [2]:
import io
from PIL import Image
import cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from huggingface_hub import notebook_login

from datasets import load_dataset, DatasetDict

from transformers import AutoImageProcessor, ViTForImageClassification

from transformers import Trainer, TrainingArguments


In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('huggingface_token') # Retrieve the token from secrets

if HF_TOKEN:
  login(HF_TOKEN)
  print("Successfully logged in to Hugging Face!")
else:
  print("Hugging Face token not found in Colab Secrets.")

Successfully logged in to Hugging Face!


#### Load Fashionpedia dataset from Huggingface

* Take a subset of the dataset as it's a large dataset.



In [4]:
from datasets import load_dataset

dataset = load_dataset('detection-datasets/fashionpedia')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-fe108070118553(…):   0%|          | 0.00/482M [00:00<?, ?B/s]

data/train-00001-of-00007-f41a5a9c38c900(…):   0%|          | 0.00/480M [00:00<?, ?B/s]

...

data/train-00005-of-00007-41d8dfe1edb659(…):   0%|          | 0.00/487M [00:00<?, ?B/s]

data/train-00006-of-00007-f41b0f2f4bbefa(…):   0%|          | 0.00/487M [00:00<?, ?B/s]

data/val-00000-of-00001-0b29e85429788213(…):   0%|          | 0.00/84.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45623 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1158 [00:00<?, ? examples/s]

In [5]:
for elem in dataset:
  print( elem, ': Number of items in dataset:-', len(dataset[elem]))

train : Number of items in dataset:- 45623
val : Number of items in dataset:- 1158


In [6]:
def create_sample(dataset, fraction = 0.1, seed = 22):
  '''
  Function generates a subset from a given dataset.
  '''
  sample_size = int(fraction*len(dataset))
  sampled_dataset = dataset.shuffle(seed = seed).select(range(sample_size))

  return sampled_dataset

In [7]:
train_dataset = create_sample(dataset['train'], fraction = 0.05)
test_dataset = create_sample(dataset['val'], fraction = 0.1)

In [8]:
print(len(train_dataset), len(test_dataset))

2281 115


In [9]:
train_dataset

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 2281
})

#### Explore dataset

* Generate ids2label and label2idx
* Explore some images to see what's in the dataset

In [ ]:
idx = 250

print('Number of detections in image id', idx)
for elem in train_dataset[idx]['objects']:
  print(elem, len(train_dataset[idx]['objects'][elem]))

In [ ]:
plt.imshow(train_dataset[idx]['image'])

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

# Put together the list of labels and ids - has to be done by hand

id2label = {
    0: 'shirt, blouse', 1: 'top, t-shirt, sweatshirt', 2: 'sweater', 3: 'cardigan',
    4: 'jacket', 5: 'vest', 6: 'pants', 7: 'shorts', 8: 'skirt', 9: 'coat',
    10: 'dress', 11: 'jumpsuit', 12: 'cape', 13: 'glasses', 14: 'hat',
    15: 'headband, head covering, hair accessory', 16: 'tie', 17: 'glove',
    18: 'watch', 19: 'belt', 20: 'leg warmer', 21: 'tights, stockings',
    22: 'sock', 23: 'shoe', 24: 'bag, wallet', 25: 'scarf', 26: 'umbrella',
    27: 'hood', 28: 'collar', 29: 'lapel', 30: 'epaulette', 31: 'sleeve',
    32: 'pocket', 33: 'neckline', 34: 'buckle', 35: 'zipper', 36: 'applique',
    37: 'bead', 38: 'bow', 39: 'flower', 40: 'fringe', 41: 'ribbon',
    42: 'rivet', 43: 'ruffle', 44: 'sequin', 45: 'tassel'
}


label2id = {v: k for k, v in id2label.items()}


In [ ]:
def draw_image_from_idx(dataset, idx):
  '''
  Function takes in image id and returns an annotated image based on image labelling.
  '''

  sample = dataset[idx]
  image = sample['image']
  objects = sample['objects']
  draw = ImageDraw.Draw(image)
  width, height = sample["width"], sample["height"]

  num_objects = len(objects['bbox_id'])

  colors = [ 'red', 'green', 'blue', 'cyan', 'yellow', 'orange', 'pink', 'purple',
            'red', 'green', 'blue', 'cyan', 'yellow', 'orange', 'pink', 'purple',
             'red', 'green', 'blue', 'cyan', 'yellow', 'orange', 'pink', 'purple']

  for obj in range(num_objects):
    box = objects['bbox'][obj]
    x1, y1, x2, y2 = tuple(box)
    draw.rectangle((x1, y1, x2, y2), outline= colors[obj], width=3)
    draw.text((x1, y1), id2label[objects["category"][obj]], fill="white",
              fontsize = 16)

  return image





In [ ]:
idx = 132
ann_image = draw_image_from_idx(train_dataset, idx)
ann_image

##### Filter invalid boxes

In [ ]:
def filter_invalid_bboxes(example):
    valid_bboxes = []
    valid_bbox_ids = []
    valid_categories = []
    valid_areas = []

    for i, bbox in enumerate(example['objects']['bbox']):
        x_min, y_min, x_max, y_max = bbox[:4]
        if x_min < x_max and y_min < y_max:
            valid_bboxes.append(bbox)
            valid_bbox_ids.append(example['objects']['bbox_id'][i])
            valid_categories.append(example['objects']['category'][i])
            valid_areas.append(example['objects']['area'][i])
        else:
            print(f"Image with invalid bbox: {example['image_id']} Invalid bbox detected and discarded: {bbox} - bbox_id: {example['objects']['bbox_id'][i]} - category: {example['objects']['category'][i]}")

    example['objects']['bbox'] = valid_bboxes
    example['objects']['bbox_id'] = valid_bbox_ids
    example['objects']['category'] = valid_categories
    example['objects']['area'] = valid_areas

    return example


In [ ]:
train_dataset = train_dataset.map(filter_invalid_bboxes)
test_dataset = test_dataset.map(filter_invalid_bboxes)

#### Next Steps

* Does one need Albumnetations
* What are the other steps before we can start with fine stuning the final layer of the model ??
